# Train a CRBM on Semeion

This notebook mirrors `examples/01_train_semeion.py`. It uses the checked-in Semeion dataset, runs one small epoch, and disables output files so it is safe for a tutorial session.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'CRBM').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from CRBM import CRBM, Trainer, load_semeion

## Model Configuration

The original Semeion experiment uses 25 bases and 20 epochs. For learning, this notebook uses fewer bases and one epoch so it finishes quickly.

In [ ]:
model_config = {
    'num_bases': 4,
    'btmup_window_shape': (5, 5),
    'block_shape': (2, 2),
    'pbias': 0.05,
    'pbias_lambda': 5,
    'init_bias': 0.01,
    'vbias': 0.001,
    'regL2': 0.01,
    'epsilon': 0.1,
    'sigma_start': 0.2,
    'sigma_stop': 0.1,
    'CD_steps': 1,
    'epoch_per_layer': 1,
    'batch': 100,
}

## Load Data

The loader returns tensors in PyTorch's `(N, C, H, W)` format after trimming the image shape for convolution and pooling.

In [ ]:
train_tensor, test_tensor = load_semeion(
    model_config['btmup_window_shape'],
    model_config['block_shape'],
)
train_tensor.shape, test_tensor.shape

## Train

`Trainer(write_outputs=False)` prevents image, error-log, and pickle output while learning the training loop.

In [ ]:
network = CRBM(
    model_config,
    (train_tensor.shape[2], train_tensor.shape[3], train_tensor.shape[1]),
)
trainer = Trainer(network, write_outputs=False)
history = trainer.fit(train_tensor, test_tensor, epochs=1)
history